# RurangaSort AI — Training Notebook

End-to-end: data acquisition -> preprocessing -> EDA -> baseline CNN -> MobileNetV2 transfer learning -> evaluation -> model selection -> save/register the winning model.

**Before running for real results:** populate `data/raw/<class>/` with the real TrashNet dataset via `python scripts/download_trashnet.py ...` (see README). The quick-start cell below falls back to a small synthetic placeholder dataset so this notebook can be smoke-tested end-to-end without network access — replace it before reporting real metrics.

In [ ]:
import sys
from pathlib import Path

ROOT_DIR = Path.cwd().parent if Path.cwd().name == "notebook" else Path.cwd()
if str(ROOT_DIR) not in sys.path:
    sys.path.insert(0, str(ROOT_DIR))

import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.config import DEFAULT_CLASS_NAMES, settings

sns.set_theme(style="whitegrid")
RAW_DIR = ROOT_DIR / settings.data_raw_dir
print("Project root:", ROOT_DIR)
print("Raw data dir:", RAW_DIR)

## 1. Data acquisition (quick-start fallback)

Skip this cell if `data/raw/<class>/` is already populated with real images.

In [ ]:
has_images = any((RAW_DIR / c).exists() and any((RAW_DIR / c).iterdir()) for c in DEFAULT_CLASS_NAMES if (RAW_DIR / c).exists())

if not has_images:
    print("No images found under data/raw/ -- generating a small synthetic placeholder dataset for smoke-testing.")
    print("Replace with the real dataset (scripts/download_trashnet.py) before reporting real results.")
    import subprocess
    subprocess.run([sys.executable, str(ROOT_DIR / "scripts" / "generate_synthetic_dataset.py"), "--count-per-class", "80"], check=True)
else:
    print("Found existing images under data/raw/ -- using them as-is.")

## 2. Data validation & preprocessing

Detect corrupted files, deduplicate, split 70/15/15 (stratified), materialize `data/train|validation|test`.

In [ ]:
from src.preprocessing import (
    build_dataset_index, deduplicate_records, split_dataset, materialize_split,
    save_class_names, save_manifest, compute_dataset_statistics,
)

records = build_dataset_index(RAW_DIR, DEFAULT_CLASS_NAMES)
print(f"Indexed {len(records)} valid (non-corrupted) images.")

records, duplicates = deduplicate_records(records)
print(f"Removed {len(duplicates)} exact duplicate images. {len(records)} remain.")

split = split_dataset(records, train_size=0.7, val_size=0.15, test_size=0.15)
for name, recs in split.items():
    print(f"  {name}: {len(recs)} images")

materialize_split(split, ROOT_DIR / "data")
save_class_names(DEFAULT_CLASS_NAMES, ROOT_DIR / settings.class_names_path)
save_manifest(split, ROOT_DIR / settings.data_processed_dir / "manifest.json",
              extra={"duplicates_removed": len(duplicates), "total_images": len(records)})
print("Saved class_names.json and processed/manifest.json")

## 3. Exploratory Data Analysis

Interpretation of (at least) three image features: class distribution, brightness, RGB colour, and dimensions/aspect ratio.

In [ ]:
stats = compute_dataset_statistics(RAW_DIR, DEFAULT_CLASS_NAMES)
stats_df = pd.DataFrame(stats).T
stats_df

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

stats_df["count"].plot(kind="bar", ax=axes[0, 0], color="steelblue")
axes[0, 0].set_title("1. Class distribution")
axes[0, 0].set_ylabel("Number of images")

stats_df["brightness_mean"].plot(kind="bar", ax=axes[0, 1], color="darkorange")
axes[0, 1].set_title("2. Mean brightness per class (0-255)")

rgb_df = pd.DataFrame({c: stats[c]["rgb_mean"] for c in DEFAULT_CLASS_NAMES if stats[c]["rgb_mean"]}).T
rgb_df.plot(kind="bar", ax=axes[1, 0], color=["red", "green", "blue"])
axes[1, 0].set_title("3. Mean RGB channel values per class")

stats_df["avg_aspect_ratio"].plot(kind="bar", ax=axes[1, 1], color="purple")
axes[1, 1].set_title("4. Average aspect ratio per class")

plt.tight_layout()
Path(ROOT_DIR / "reports" / "figures").mkdir(parents=True, exist_ok=True)
plt.savefig(ROOT_DIR / "reports" / "figures" / "eda_overview.png", dpi=120)
plt.show()

**Interpretation**

1. **Class distribution** — an uneven bar chart flags class imbalance; the model can learn to over-predict a majority class, which is why macro F1 (not accuracy) is used for model selection below.
2. **Brightness** — large differences between classes usually reflect lighting/background differences in how each class was photographed rather than a true material property; that's a shortcut the CNN could latch onto instead of shape.
3. **RGB channel means** — a dominant channel per class (e.g. browner cardboard, cooler glass/metal) shows the model may partly rely on colour, which won't generalise to unusually coloured items of the same material.
4. **Aspect ratio** — wide spread suggests photos came from different sources/crops; squashing everything to 224x224 can distort elongated items (e.g. bottles), a useful lead if that class underperforms in the confusion matrix later.

## 4. Load train/validation/test datasets

In [ ]:
from src.training import load_datasets

train_ds, val_ds, test_ds, class_names = load_datasets(
    ROOT_DIR / settings.data_train_dir, ROOT_DIR / settings.data_validation_dir, ROOT_DIR / settings.data_test_dir,
    batch_size=16,
)
print("Class names:", class_names)

## 5. Baseline CNN

In [ ]:
import tensorflow as tf
from src.model import build_baseline_cnn

baseline_model = build_baseline_cnn(num_classes=len(class_names))
baseline_model.summary()

baseline_history = baseline_model.fit(
    train_ds, validation_data=val_ds, epochs=10,
    callbacks=[tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=4, restore_best_weights=True)],
)

## 6. MobileNetV2 (transfer learning, two-stage)

In [ ]:
from src.model import build_mobilenet_v2, unfreeze_for_fine_tuning

mobilenet_model = build_mobilenet_v2(num_classes=len(class_names))
mobilenet_history_head = mobilenet_model.fit(
    train_ds, validation_data=val_ds, epochs=8,
    callbacks=[tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=3, restore_best_weights=True)],
)

mobilenet_model = unfreeze_for_fine_tuning(mobilenet_model, learning_rate=1e-5)
mobilenet_history_fine_tune = mobilenet_model.fit(
    train_ds, validation_data=val_ds, epochs=5,
    callbacks=[tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=3, restore_best_weights=True)],
)

## 7. Training curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(baseline_history.history["accuracy"], label="train")
axes[0].plot(baseline_history.history["val_accuracy"], label="validation")
axes[0].set_title("Baseline CNN accuracy")
axes[0].legend()

mn_acc = mobilenet_history_head.history["accuracy"] + mobilenet_history_fine_tune.history["accuracy"]
mn_val_acc = mobilenet_history_head.history["val_accuracy"] + mobilenet_history_fine_tune.history["val_accuracy"]
axes[1].plot(mn_acc, label="train")
axes[1].plot(mn_val_acc, label="validation")
axes[1].axvline(len(mobilenet_history_head.history["accuracy"]) - 0.5, color="grey", linestyle="--", label="fine-tune starts")
axes[1].set_title("MobileNetV2 accuracy")
axes[1].legend()
plt.tight_layout()
plt.savefig(ROOT_DIR / "reports" / "figures" / "training_curves.png", dpi=120)
plt.show()

## 8. Evaluation — full metric suite for both models

In [ ]:
from src.training import save_trained_model
from src.evaluation import evaluate_model

baseline_path = save_trained_model(baseline_model, class_names, ROOT_DIR / "models" / "_notebook_baseline", "baseline_cnn")
mobilenet_path = save_trained_model(mobilenet_model, class_names, ROOT_DIR / "models" / "_notebook_mobilenet", "mobilenet_v2")

baseline_metrics = evaluate_model(baseline_model, test_ds, class_names, model_path=baseline_path)
mobilenet_metrics = evaluate_model(mobilenet_model, test_ds, class_names, model_path=mobilenet_path)

comparison = pd.DataFrame({
    "baseline_cnn": {
        "test_accuracy": baseline_metrics["accuracy"],
        "macro_precision": baseline_metrics["macro_precision"],
        "macro_recall": baseline_metrics["macro_recall"],
        "macro_f1": baseline_metrics["macro_f1"],
        "weighted_f1": baseline_metrics["weighted_f1"],
        "roc_auc": baseline_metrics["roc_auc_ovr_macro"],
        "pr_auc": baseline_metrics["pr_auc_macro"],
        "log_loss": baseline_metrics["log_loss"],
        "avg_latency_ms": baseline_metrics["avg_latency_ms"],
        "p95_latency_ms": baseline_metrics["p95_latency_ms"],
        "model_size_mb": baseline_metrics.get("model_size_mb"),
    },
    "mobilenet_v2": {
        "test_accuracy": mobilenet_metrics["accuracy"],
        "macro_precision": mobilenet_metrics["macro_precision"],
        "macro_recall": mobilenet_metrics["macro_recall"],
        "macro_f1": mobilenet_metrics["macro_f1"],
        "weighted_f1": mobilenet_metrics["weighted_f1"],
        "roc_auc": mobilenet_metrics["roc_auc_ovr_macro"],
        "pr_auc": mobilenet_metrics["pr_auc_macro"],
        "log_loss": mobilenet_metrics["log_loss"],
        "avg_latency_ms": mobilenet_metrics["avg_latency_ms"],
        "p95_latency_ms": mobilenet_metrics["p95_latency_ms"],
        "model_size_mb": mobilenet_metrics.get("model_size_mb"),
    },
})
comparison

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
sns.heatmap(baseline_metrics["confusion_matrix"], annot=True, fmt="d", xticklabels=class_names, yticklabels=class_names, ax=axes[0], cmap="Blues")
axes[0].set_title("Baseline CNN — confusion matrix")
sns.heatmap(mobilenet_metrics["confusion_matrix"], annot=True, fmt="d", xticklabels=class_names, yticklabels=class_names, ax=axes[1], cmap="Greens")
axes[1].set_title("MobileNetV2 — confusion matrix")
plt.tight_layout()
plt.savefig(ROOT_DIR / "reports" / "figures" / "confusion_matrices.png", dpi=120)
plt.show()

## 9. Model selection

Selection is based on **macro F1** (equal weight per class) rather than raw accuracy, since the dataset is imbalanced (see EDA above). Latency and model size are tie-breakers when macro F1 is close.

In [ ]:
selected_name = "mobilenet_v2" if mobilenet_metrics["macro_f1"] >= baseline_metrics["macro_f1"] else "baseline_cnn"
selected_model = mobilenet_model if selected_name == "mobilenet_v2" else baseline_model
selected_metrics = mobilenet_metrics if selected_name == "mobilenet_v2" else baseline_metrics
selected_path = mobilenet_path if selected_name == "mobilenet_v2" else baseline_path

print(f"Selected model: {selected_name}")
print(f"Macro F1: {selected_metrics['macro_f1']:.4f}")
print(f"Test accuracy: {selected_metrics['accuracy']:.4f}")
print(f"Avg latency: {selected_metrics['avg_latency_ms']:.1f} ms")

## 10. Save & register the selected model as the active model

In [ ]:
from src.model_registry import register_candidate, evaluate_promotion, promote_candidate, get_active_metrics, reject_candidate

register_candidate(
    model_file=selected_path, class_names=class_names, metrics=selected_metrics,
    model_name=selected_name, input_shape=settings.image_shape,
)

active_metrics = get_active_metrics()
decision = evaluate_promotion(selected_metrics, active_metrics)
print(decision["reasons"])

if decision["should_promote"]:
    metadata = promote_candidate()
    print("Promoted to active model:", metadata["model_version"])
else:
    reject_candidate(decision["reasons"])
    print("Candidate rejected — active model unchanged.")

## 11. Prediction function demo

In [ ]:
from src.prediction import Predictor

predictor = Predictor()
predictor.load()

sample_class = class_names[0]
sample_dir = ROOT_DIR / settings.data_test_dir / sample_class
sample_path = next(sample_dir.iterdir())

with open(sample_path, "rb") as fh:
    sample_bytes = fh.read()

result = predictor.predict(sample_bytes)
print(f"True class: {sample_class}")
print(json.dumps(result, indent=2))

## Next steps

- Replace the synthetic dataset with the real TrashNet dataset and re-run this notebook end-to-end.
- Paste the resulting `comparison` table and confusion matrices into the README's [Model Results](../README.md#model-results) section.
- Start the API (`uvicorn api.main:app`) and UI (`streamlit run ui/app.py`) to exercise prediction, upload, and retraining through the full system.